# Análise de dados TCP-CI

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('./T CELL/DENV 3 - T Cell Prediction - Class I.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,VSGKLIHEW,303,311,9,HLA-B*57:01,303,0.01,VSGKLIHEW,VSGKLIHEW,0.997373,0.01
1,1,VSGKLIHEW,303,311,9,HLA-B*58:01,303,0.01,VSGKLIHEW,VSGKLIHEW,0.996354,0.01
2,1,TTRMENLLW,60,68,9,HLA-B*57:01,60,0.01,TTRMENLLW,TTRMENLLW,0.990271,0.01
3,1,CTWPKSHTLW,223,232,10,HLA-B*57:01,567,0.01,CTWPKSHTW,CTWPKSHTLW,0.989558,0.01
4,1,EVHTWTEQY,24,32,9,HLA-A*26:01,24,0.01,EVHTWTEQY,EVHTWTEQY,0.988494,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
36985,1,PISEKEENMVKS,337,348,12,HLA-A*24:02,1366,100.00,PISEKMVKS,PISEKEENMVKS,0.000000,100.00
36986,1,PISEKEENMVKS,337,348,12,HLA-A*32:01,1366,100.00,PISENMVKS,PISEKEENMVKS,0.000000,100.00
36987,1,PISEKEENMVKS,337,348,12,HLA-B*35:01,1366,100.00,SEKEENVKS,SEKEENMVKS,0.000000,100.00
36988,1,PISEKEENMVKS,337,348,12,HLA-B*53:01,1366,100.00,PISEKEVKS,PISEKEENMVKS,0.000000,100.00


## Selecionando Epítopos com median binding percentile menor que 5.

In [3]:
df_mbp_m5 = df[df['median binding percentile'] < 5].copy()
df_mbp_m5

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,VSGKLIHEW,303,311,9,HLA-B*57:01,303,0.01,VSGKLIHEW,VSGKLIHEW,0.997373,0.01
1,1,VSGKLIHEW,303,311,9,HLA-B*58:01,303,0.01,VSGKLIHEW,VSGKLIHEW,0.996354,0.01
2,1,TTRMENLLW,60,68,9,HLA-B*57:01,60,0.01,TTRMENLLW,TTRMENLLW,0.990271,0.01
3,1,CTWPKSHTLW,223,232,10,HLA-B*57:01,567,0.01,CTWPKSHTW,CTWPKSHTLW,0.989558,0.01
4,1,EVHTWTEQY,24,32,9,HLA-A*26:01,24,0.01,EVHTWTEQY,EVHTWTEQY,0.988494,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
2604,1,QPMELKYSW,107,115,9,HLA-B*40:01,107,4.90,QPMELKYSW,QPMELKYSW,0.002655,4.90
2605,1,MELKYSWKTW,109,118,10,HLA-B*40:01,453,4.90,MELKYSKTW,MELKYSWKTW,0.002635,4.90
2606,1,SWKLEKASLIEV,209,220,12,HLA-B*40:01,1238,4.90,SEKASLIEV,SWKLEKASLIEV,0.002630,4.90
2607,1,TESCGTRGP,288,296,9,HLA-B*40:01,288,4.90,TESCGTRGP,TESCGTRGP,0.002597,4.90


## Agrupando por pepitideos e agregando colunas pertinentes.

In [4]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    ))

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAVKDERAV,186,194,2,3.10,"HLA-A*02:06, HLA-B*51:01"
1,AAVKDERAVH,186,195,2,4.00,"HLA-A*30:02, HLA-B*15:01"
2,ADMGYWIESQK,196,206,2,3.15,"HLA-A*03:01, HLA-A*11:01"
3,ADSPKRLATAI,36,46,1,0.52,HLA-B*07:02
4,AETQNSSFI,126,134,3,0.25,"HLA-B*40:01, HLA-B*44:02, HLA-B*44:03"
...,...,...,...,...,...,...
599,YTQLCDHRL,175,183,9,3.70,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:06, HLA-A*2..."
600,YTQLCDHRLM,175,184,3,3.40,"HLA-A*01:01, HLA-B*57:01, HLA-B*58:01"
601,YTQLCDHRLMSA,175,186,1,2.50,HLA-A*01:01
602,YWIESQKNGSW,200,210,9,1.40,"HLA-A*23:01, HLA-A*24:02, HLA-A*32:01, HLA-B*4..."


# Filtragem por epítopos presentes em mais de determinada quantidade de alelos.

In [5]:
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= 10
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AVHADMGYW,193,201,10,1.645,"HLA-A*23:01, HLA-A*26:01, HLA-A*30:02, HLA-A*3..."
1,CTLPPLRYM,316,324,20,3.250,"HLA-A*01:01, HLA-A*02:06, HLA-A*03:01, HLA-A*1..."
2,CTWPKSHTL,223,231,23,1.200,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
3,ECPSASRAW,142,150,10,2.150,"HLA-A*23:01, HLA-A*24:02, HLA-A*26:01, HLA-B*3..."
4,ELKYSWKTW,110,118,11,1.500,"HLA-A*23:01, HLA-A*24:02, HLA-A*26:01, HLA-A*3..."
5,EVHTWTEQY,24,32,16,1.500,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
6,EVHTWTEQYKF,24,34,10,3.250,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
7,FQADSPKRL,34,42,18,2.450,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."
8,FTTNIWLKL,163,171,16,1.650,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
9,FVTNEVHTW,20,28,17,1.200,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:06, HLA-A*2..."


In [6]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["qte_de_alelos", "median_binding_percentile"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,CTWPKSHTL,223,231,23,1.200,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
1,HTWTEQYKF,26,34,21,1.500,"HLA-A*01:01, HLA-A*02:06, HLA-A*11:01, HLA-A*2..."
2,TLTPQPMEL,103,111,20,1.900,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
3,CTLPPLRYM,316,324,20,3.250,"HLA-A*01:01, HLA-A*02:06, HLA-A*03:01, HLA-A*1..."
4,KQIANELNY,69,77,19,1.400,"HLA-A*01:01, HLA-A*02:06, HLA-A*03:01, HLA-A*1..."
5,KLREVYTQL,170,178,18,1.400,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*0..."
6,FQADSPKRL,34,42,18,2.450,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."
7,FVTNEVHTW,20,28,17,1.200,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:06, HLA-A*2..."
8,TPQPMELKY,105,113,17,1.400,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
9,TCTWPKSHTL,222,231,17,2.100,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
